1. Create an input text file consisting of continuous, meaningful English text of approximately 10 pages. The text may be collected from books or articles and should be free of tables, code, or bullet points.

2. Load the input text and perform necessary preprocessing steps such as sentence segmentation, tokenization, and case folding.

In [1]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter, defaultdict
import random
import numpy as np

with open('text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

sentences = sent_tokenize(text)
tokens = [word.lower() for word in word_tokenize(text) if word.isalpha()]

print(f"Total sentences: {len(sentences)}")
print(f"Total tokens: {len(tokens)}")
print(f"Sample tokens: {tokens[:20]}")

Total sentences: 221
Total tokens: 7520
Sample tokens: ['to', 'the', 'door', 'of', 'an', 'inn', 'in', 'the', 'provincial', 'town', 'of', 'there', 'drew', 'up', 'a', 'smart', 'light', 'of', 'the', 'sort']


3. From the preprocessed text, compute the frequency counts for:

- Unigrams
- Bi-grams
- Tri-grams

In [ ]:
unigrams = tokens
bigrams = list(zip(tokens[:-1], tokens[1:]))
trigrams = list(zip(tokens[:-2], tokens[1:-1], tokens[2:]))

unigram_freq = Counter(unigrams)
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

print(f"Unique unigrams: {len(unigram_freq)}")
print(f"Unique bigrams: {len(bigram_freq)}")
print(f"Unique trigrams: {len(trigram_freq)}")
print()
print(f"Top 10 unigrams: {unigram_freq.most_common(10)}")
print(f"Top 10 bigrams: {bigram_freq.most_common(10)}")
print(f"Top 10 trigrams: {trigram_freq.most_common(10)}")

Unique unigrams: 2082
Unique bigrams: 5838
Unique trigrams: 7212

Top 10 unigrams: [('the', 578), ('of', 361), ('a', 278), ('to', 241), ('and', 233), ('in', 144), ('his', 127), ('he', 126), ('with', 99), ('that', 96)]

Top 10 bigrams: [(('of', 'the'), 93), (('in', 'the'), 39), (('to', 'the'), 37), (('to', 'be'), 33), (('of', 'a'), 24), (('and', 'the'), 23), (('with', 'a'), 20), (('on', 'the'), 20), (('he', 'had'), 20), (('for', 'the'), 19)]

Top 10 trigrams: [(('of', 'the', 'local'), 9), (('a', 'pair', 'of'), 8), (('the', 'local', 'council'), 8), (('president', 'of', 'the'), 7), (('the', 'chief', 'of'), 7), (('a', 'couple', 'of'), 6), (('happened', 'to', 'be'), 6), (('and', 'so', 'forth'), 6), (('that', 'he', 'was'), 6), (('the', 'president', 'of'), 6)]


4. Using the computed counts, estimate the probability distributions for:

- Unigram language model
- Bi-gram language model
- Tri-gram language model

Clearly specify the probability equations used for each model.

In [3]:
total_unigrams = len(unigrams)
vocab_size = len(unigram_freq)

unigram_prob = {word: count / total_unigrams for word, count in unigram_freq.items()}

bigram_prob = defaultdict(dict)
for (w1, w2), count in bigram_freq.items():
    bigram_prob[w1][w2] = count / unigram_freq[w1]

trigram_prob = defaultdict(lambda: defaultdict(dict))
for (w1, w2, w3), count in trigram_freq.items():
    trigram_prob[w1][w2][w3] = count / bigram_freq[(w1, w2)]

print("Probability Equations:")
print("Unigram: P(w) = count(w) / total_words")
print("Bigram: P(w2|w1) = count(w1,w2) / count(w1)")
print("Trigram: P(w3|w1,w2) = count(w1,w2,w3) / count(w1,w2)")
print(f"\nSample unigram probabilities: {list(unigram_prob.items())[:5]}")
print(f"Sample bigram probability P('door'|'the'): {bigram_prob.get('the', {}).get('door', 0):.4f}")

Probability Equations:
Unigram: P(w) = count(w) / total_words
Bigram: P(w2|w1) = count(w1,w2) / count(w1)
Trigram: P(w3|w1,w2) = count(w1,w2,w3) / count(w1,w2)

Sample unigram probabilities: [('to', 0.03204787234042553), ('the', 0.07686170212765958), ('door', 0.0005319148936170213), ('of', 0.048005319148936174), ('an', 0.0035904255319148936)]
Sample bigram probability P('door'|'the'): 0.0052


5. Using each of the trained models separately, generate a paragraph consisting of 5 meaningful sentences using:

- Unigram model
- Bi-gram model
- Tri-gram model

In [5]:
def generate_unigram(n_words=50):
    words = list(unigram_prob.keys())
    probs = list(unigram_prob.values())
    return ' '.join(np.random.choice(words, size=n_words, p=probs))

def generate_bigram(n_words=50):
    current = random.choice(list(bigram_prob.keys()))
    result = [current]
    for _ in range(n_words - 1):
        if current in bigram_prob and bigram_prob[current]:
            next_words = list(bigram_prob[current].keys())
            probs = list(bigram_prob[current].values())
            probs = np.array(probs)
            probs = probs / probs.sum()
            current = np.random.choice(next_words, p=probs)
            result.append(current)
        else:
            current = random.choice(list(bigram_prob.keys()))
            result.append(current)
    return ' '.join(result)

def generate_trigram(n_words=50):
    w1, w2 = random.choice(list(bigram_freq.keys()))
    result = [w1, w2]
    for _ in range(n_words - 2):
        if w1 in trigram_prob and w2 in trigram_prob[w1] and trigram_prob[w1][w2]:
            next_words = list(trigram_prob[w1][w2].keys())
            probs = list(trigram_prob[w1][w2].values())
            probs = np.array(probs)
            probs = probs / probs.sum()
            w3 = np.random.choice(next_words, p=probs)
            result.append(w3)
            w1, w2 = w2, w3
        else:
            w1, w2 = random.choice(list(bigram_freq.keys()))
            result.extend([w1, w2])
    return ' '.join(result)

print("Unigram Generated Text:")
print(generate_unigram())
print("\nBigram Generated Text:")
print(generate_bigram())
print("\nTrigram Generated Text:")
print(generate_trigram())

Unigram Generated Text:
hair loud dingier to apartment up desire is bestow the named has brown with to the in fully is he than you and the having local and time director the that the that the yet plays the while the newcomer to his souls brief the that state in his short

Bigram Generated Text:
add that according to be situated and more than fifteen versts the remainder were being got soup a post in recognition of the britchka leapt forward with a dish of the public prosecutor the gentleman lives in their governor he hastened to say something of this occasion chichikov toes and

Trigram Generated Text:
than do the most ample meed of praise again to return with other predatory squadrons indeed so dazed was chichikov that scarcely did he say outright you played the following day he devoted to paying calls upon the fulfilling of it as a sacred duty in the town and lastly


6. Compare the grammatical correctness and contextual coherence of the text generated by the three models.

Unigram model:
- Grammar: Very poor, words are randomly selected
- Coherence: No contextual meaning, completely random
- Output resembles: Random word salad

Bigram model:
- Grammar: Better, some local structure preserved
- Coherence: Limited context (1 word history)
- Output resembles: Short phrases that make partial sense

Trigram model:
- Grammar: Best, maintains local grammatical structure
- Coherence: Better context (2 word history)
- Output resembles: More coherent sentences with better flow

Conclusion: Higher-order n-grams capture more context and produce more coherent text

7. Split the dataset into training and test portions.

In [7]:
split_idx = int(0.8 * len(tokens))
train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]

train_unigrams = train_tokens
train_bigrams = list(zip(train_tokens[:-1], train_tokens[1:]))
train_trigrams = list(zip(train_tokens[:-2], train_tokens[1:-1], train_tokens[2:]))

train_unigram_freq = Counter(train_unigrams)
train_bigram_freq = Counter(train_bigrams)
train_trigram_freq = Counter(train_trigrams)

print(f"Training tokens: {len(train_tokens)}")
print(f"Test tokens: {len(test_tokens)}")
print(f"Train/Test split: 80/20")

Training tokens: 6016
Test tokens: 1504
Train/Test split: 80/20


8. Evaluate the perplexity of the unigram, bi-gram, and tri-gram models on the test data.

In [8]:
def calculate_perplexity_unigram(test_tokens, train_freq):
    total_train = sum(train_freq.values())
    vocab_size = len(train_freq)
    log_prob = 0
    for word in test_tokens:
        prob = (train_freq.get(word, 0) + 1) / (total_train + vocab_size)
        log_prob += np.log2(prob)
    return 2 ** (-log_prob / len(test_tokens))

def calculate_perplexity_bigram(test_tokens, train_unigram_freq, train_bigram_freq):
    vocab_size = len(train_unigram_freq)
    log_prob = 0
    for i in range(len(test_tokens) - 1):
        w1, w2 = test_tokens[i], test_tokens[i+1]
        count_w1 = train_unigram_freq.get(w1, 0)
        count_w1_w2 = train_bigram_freq.get((w1, w2), 0)
        prob = (count_w1_w2 + 1) / (count_w1 + vocab_size)
        log_prob += np.log2(prob)
    return 2 ** (-log_prob / (len(test_tokens) - 1))

def calculate_perplexity_trigram(test_tokens, train_bigram_freq, train_trigram_freq):
    vocab_size = len(set(test_tokens))
    log_prob = 0
    for i in range(len(test_tokens) - 2):
        w1, w2, w3 = test_tokens[i], test_tokens[i+1], test_tokens[i+2]
        count_w1_w2 = train_bigram_freq.get((w1, w2), 0)
        count_w1_w2_w3 = train_trigram_freq.get((w1, w2, w3), 0)
        prob = (count_w1_w2_w3 + 1) / (count_w1_w2 + vocab_size)
        log_prob += np.log2(prob)
    return 2 ** (-log_prob / (len(test_tokens) - 2))

perplexity_unigram = calculate_perplexity_unigram(test_tokens, train_unigram_freq)
perplexity_bigram = calculate_perplexity_bigram(test_tokens, train_unigram_freq, train_bigram_freq)
perplexity_trigram = calculate_perplexity_trigram(test_tokens, train_bigram_freq, train_trigram_freq)

print(f"Unigram Perplexity: {perplexity_unigram:.2f}")
print(f"Bigram Perplexity: {perplexity_bigram:.2f}")
print(f"Trigram Perplexity: {perplexity_trigram:.2f}")

Unigram Perplexity: 636.43
Bigram Perplexity: 1275.71
Trigram Perplexity: 641.31


9. Analyze the perplexity values obtained and explain how increasing the order of the n-gram model affects language modeling performance

Results:
- Unigram: 636.43 (BEST - lowest perplexity)
- Trigram: 641.31 
- Bigram: 1275.71 (WORST - highest perplexity)

Observations:
1. Lower perplexity = Better model performance
2. UNIGRAM has lowest perplexity (best on this test set)
3. Bigram performs worst with highest perplexity
4. Trigram is in between

Explanation:
This counterintuitive result occurs because:
- Dataset is relatively small (7520 tokens)
- Bigram model suffers from data sparsity issues
- Add-1 smoothing doesn't work well with sparse bigram counts
- Many bigrams in test set were not seen in training
- Unigram model is more robust to unseen data with this smoothing

Typically with larger datasets, bigram and trigram would outperform unigram as they capture more context.